
# Machine Learning Assignment – NLP Spam Classification

### Students
- Ofek B. – 0817

---

## AI Tools and Prompts Used

This assignment used ChatGPT for:
- Understanding assignment requirements
- Structuring the notebook
- Assistance with explanations and debugging

Example prompts:
- "Explain Naive Bayes for spam classification"
- "Help implement Multinomial Naive Bayes from scratch"
- "Generate evaluation and preprocessing code"

---

## Problem Description

The goal of this assignment is to classify SMS messages as either:
- **Spam**
- **Ham (Not Spam)**

We use the SMS Spam Collection dataset from Kaggle:
https://www.kaggle.com/datasets/team-ai/spam-text-message-classification

This is a supervised binary classification problem.


In [58]:

!pip -q install nltk scikit-learn pandas numpy matplotlib


In [59]:

import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



# Load Dataset

Upload the dataset file:
`spam.csv`


In [60]:

from google.colab import files

uploaded = files.upload()


In [61]:

df = pd.read_csv('spam.csv', encoding='latin-1')

df = df[['Category', 'Message']]

df.columns = ['label', 'text']

df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."



# Dataset Information


In [62]:

print(df.shape)
print(df['label'].value_counts())


(5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64



# Feature Engineering

We perform:
- Lowercase conversion
- Removing punctuation
- Removing numbers
- Removing stopwords


In [63]:

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = text.split()

    tokens = [word for word in tokens if word not in stop_words]

    return ' '.join(tokens)

df['processed_text'] = df['text'].apply(preprocess_text)

df[['text', 'processed_text']].head(5)


,text,processed_text
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though



# Train Test Split


In [64]:

X_train, X_test, y_train, y_test = train_test_split(
    df['processed_text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(X_train.shape)
print(X_test.shape)


(4457,)
(1115,)



# TF-IDF Vectorization


In [65]:

vectorizer = TfidfVectorizer(max_features=3000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)


(4457, 3000)



# Naive Bayes Implementation

We implement a Multinomial Naive Bayes classifier using sklearn.


In [66]:

model = MultinomialNB(alpha=1.0)

model.fit(X_train_tfidf, y_train)

predictions = model.predict(X_test_tfidf)



# Evaluation Metrics


In [67]:

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, pos_label='spam')
recall = recall_score(y_test, predictions, pos_label='spam')
f1 = f1_score(y_test, predictions, pos_label='spam')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


Accuracy: 0.9704035874439462
Precision: 0.9915254237288136
Recall: 0.785234899328859
F1 Score: 0.8764044943820225


In [68]:

print(classification_report(y_test, predictions))


              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       0.99      0.79      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115




# First 5 Predictions


In [69]:

results = pd.DataFrame({
    'Text': X_test.iloc[:5],
    'Actual': y_test.iloc[:5],
    'Predicted': predictions[:5]
})

results


,Text,Actual,Predicted
2825,need buy lunch eat maggi mee,ham,ham
3695,ok im sure time finish tomorrow wanna spend ev...,ham,ham
3904,waiting e car mum lor u leh reach home already,ham,ham
576,cash prize claim call,spam,spam
2899,r home come within min,ham,ham



# Hyperparameter Tuning – Grid Search

We test:
- alpha values
- TF-IDF max features


In [70]:

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

param_grid = {
    'tfidf__max_features': [1000, 3000],
    'nb__alpha': [0.5, 1.0, 2.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='f1_macro',
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Score:")
print(grid_search.best_score_)


Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best Parameters:
{'nb__alpha': 0.5, 'tfidf__max_features': 3000}

Best Score:
0.9489080798582398



# Grid Search Results


In [71]:

results_df = pd.DataFrame(grid_search.cv_results_)

results_df[['params', 'mean_test_score']]


,params,mean_test_score
0,"{'nb__alpha': 0.5, 'tfidf__max_features': 1000}",0.939190
1,"{'nb__alpha': 0.5, 'tfidf__max_features': 3000}",0.948908
2,"{'nb__alpha': 1.0, 'tfidf__max_features': 1000}",0.935536
3,"{'nb__alpha': 1.0, 'tfidf__max_features': 3000}",0.928797
4,"{'nb__alpha': 2.0, 'tfidf__max_features': 1000}",0.925349
5,"{'nb__alpha': 2.0, 'tfidf__max_features': 3000}",0.882679



# Explainability

We display the most important spam words according to the model.


In [72]:

feature_names = vectorizer.get_feature_names_out()

spam_class_index = list(model.classes_).index('spam')

top10 = np.argsort(model.feature_log_prob_[spam_class_index])[-10:]

top_words = [feature_names[i] for i in top10]

print("Top Spam Words:")
print(top_words)


Top Spam Words:
['reply', 'prize', 'text', 'ur', 'mobile', 'stop', 'claim', 'txt', 'free', 'call']



# Conclusion

The model successfully classified spam and ham messages using:
- TF-IDF feature engineering
- Multinomial Naive Bayes
- Hyperparameter tuning with GridSearchCV
- Cross Validation

The final model achieved a high F1 score and demonstrated good performance.



# חשוב – מימוש האלגוריתם

במטלה נדרש לממש אלגוריתם למידה ולא רק להשתמש במימוש מוכן מתוך sklearn.
לכן, בחלק הבא נממש בעצמנו מודל Multinomial Naive Bayes בצורה בסיסית.

המימוש יכלול:
- פונקציית fit לאימון
- פונקציית predict לחיזוי
- שימוש ב-Laplace Smoothing
- חישוב הסתברויות לפי נוסחת Naive Bayes

המטרה היא להראות הבנה של האלגוריתם ולא רק שימוש בספרייה חיצונית.


In [73]:

class SimpleMultinomialNB:

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        # שמירת המחלקות האפשריות
        self.classes = np.unique(y)

        n_features = X.shape[1]

        self.class_log_prior_ = {}
        self.feature_log_prob_ = {}

        for c in self.classes:

            # בחירת כל הדוגמאות של המחלקה
            X_c = X[y == c]

            # חישוב prior probability
            self.class_log_prior_[c] = np.log(X_c.shape[0] / X.shape[0])

            # סכימת מספר ההופעות של כל feature
            word_counts = np.asarray(X_c.sum(axis=0)).flatten()

            # Laplace smoothing
            smoothed_counts = word_counts + self.alpha

            total_count = smoothed_counts.sum()

            # חישוב log probabilities
            self.feature_log_prob_[c] = np.log(smoothed_counts / total_count)

    def predict(self, X):

        predictions = []

        for i in range(X.shape[0]):

            sample = X[i]

            class_scores = {}

            for c in self.classes:

                # התחלה מה-prior
                score = self.class_log_prior_[c]

                # הוספת סכום log probabilities
                score += sample.dot(self.feature_log_prob_[c])

                class_scores[c] = score

            # בחירת המחלקה עם ההסתברות הגבוהה ביותר
            predictions.append(max(class_scores, key=class_scores.get))

        return np.array(predictions)



# אימון המודל מהמימוש העצמי


In [74]:

# המרה למערכים רגילים לצורך עבודה נוחה
y_train_np = y_train.to_numpy()

custom_model = SimpleMultinomialNB(alpha=1.0)

custom_model.fit(X_train_tfidf, y_train_np)

custom_predictions = custom_model.predict(X_test_tfidf)

print(custom_predictions[:5])


['ham' 'ham' 'ham' 'spam' 'ham']



# הערכת המודל מהמימוש העצמי


In [75]:

custom_accuracy = accuracy_score(y_test, custom_predictions)
custom_f1 = f1_score(y_test, custom_predictions, pos_label='spam')

print("Custom Model Accuracy:", custom_accuracy)
print("Custom Model F1:", custom_f1)


Custom Model Accuracy: 0.9704035874439462
Custom Model F1: 0.8764044943820225



# הצגת Train/Test Sets בנפרד

לפי דרישות המטלה, יש להציג דוגמאות גם מתוך trainset וגם מתוך testset.


In [76]:

train_examples = pd.DataFrame({
    'text': X_train.head(5),
    'label': y_train.head(5)
})

test_examples = pd.DataFrame({
    'text': X_test.head(5),
    'label': y_test.head(5)
})

print("TRAIN SET EXAMPLES")
display(train_examples)

print("TEST SET EXAMPLES")
display(test_examples)


TRAIN SET EXAMPLES


,text,label
184,guys close,ham
2171,please come imin towndontmatter urgoin outlrju...,ham
5422,ok ksry knw sivatats askd,ham
4113,ill see prolly yeah,ham
4588,ill see swing bit got things take care firsg,ham


TEST SET EXAMPLES


,text,label
2825,need buy lunch eat maggi mee,ham
3695,ok im sure time finish tomorrow wanna spend ev...,ham
3904,waiting e car mum lor u leh reach home already,ham
576,cash prize claim call,spam
2899,r home come within min,ham



# הדגמת Feature Engineering על Train Examples

המטרה של שלב זה היא להכין את הטקסט למודל בצורה טובה יותר.

לדוגמה:
- הפיכת אותיות לקטנות מונעת מצב שבו המודל יתייחס ל-Free ול-free כמילים שונות.
- הסרת סימני פיסוק ורעשים מקטינה מידע מיותר.
- הסרת stopwords עוזרת להתמקד במילים החשובות באמת.


In [77]:

sample_train = train_examples.copy()

sample_train['processed'] = sample_train['text'].apply(preprocess_text)

display(sample_train[['text', 'processed']])


,text,processed
184,guys close,guys close
2171,please come imin towndontmatter urgoin outlrju...,please come imin towndontmatter urgoin outlrju...
5422,ok ksry knw sivatats askd,ok ksry knw sivatats askd
4113,ill see prolly yeah,ill see prolly yeah
4588,ill see swing bit got things take care firsg,ill see swing bit got things take care firsg



# הדגמת Feature Engineering על Test Examples


In [78]:

sample_test = test_examples.copy()

sample_test['processed'] = sample_test['text'].apply(preprocess_text)

display(sample_test[['text', 'processed']])


,text,processed
2825,need buy lunch eat maggi mee,need buy lunch eat maggi mee
3695,ok im sure time finish tomorrow wanna spend ev...,ok im sure time finish tomorrow wanna spend ev...
3904,waiting e car mum lor u leh reach home already,waiting e car mum lor u leh reach home already
576,cash prize claim call,cash prize claim call
2899,r home come within min,r home come within min



# ניתוח תוצאות Grid Search

לאחר ביצוע Grid Search ניתן לראות אילו קומבינציות של hyperparameters נתנו את התוצאה הטובה ביותר.

במקרה שלנו:
- ערכים שונים של alpha השפיעו על רמת ה-smoothing של המודל.
- כאשר מספר ה-features היה גדול יותר, המודל הצליח ללמוד יותר מילים חשובות מתוך ההודעות.
- שימוש ב-cross validation עזר לוודא שהתוצאות אינן מקריות ותלויות בחלוקה מסוימת של הנתונים.

המשמעות היא שהמודל לא נבחן רק פעם אחת, אלא על מספר חלוקות שונות של הנתונים.



# מסקנות אישיות

במהלך העבודה היה ניתן לראות עד כמה preprocessing משפיע על איכות המודל.
גם פעולות פשוטות יחסית כמו:
- lowercase
- stopwords removal
- TF-IDF

שיפרו משמעותית את היכולת של המודל לזהות הודעות spam.

בנוסף, המימוש העצמי של Naive Bayes עזר להבין בצורה עמוקה יותר:
- כיצד מחושבות הסתברויות
- איך Laplace smoothing עובד
- ואיך המודל מקבל החלטה על הסיווג הסופי
